# Filter Designer

Enter one lowpass specification, tick the filter types to compare, and each is designed at the smallest size that meets the spec.

Keep `filter_designer.py` in the same folder. Needs `numpy`, `scipy`, `matplotlib` and `ipywidgets`.

In [ ]:
# %pip install numpy scipy matplotlib ipywidgets   # uncomment if anything is missing
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import filter_designer as fd

## 1. Interactive designer

- **Specification.** Frequencies are in whatever unit you use for the sample rate. Ripple is peak to peak.
- **Filter types.** Tick any number. Colours stay with the type.
- **Quantise coefficients.** Draws the response with the given word length as a dashed line, and the table switches to the quantised figures.
- **Input to the filter.** Adds a plot of the input spectrum before and after each filter, plus signal loss, adjacent-channel rejection and EVM where they apply. Bandwidth is two-sided: the symbol rate for RRC and raised cosine.
- **Export coefficients.** Writes files for every ticked filter into the export folder.

How to read the multiplier columns:

| Column | Meaning |
|---|---|
| every non-zero coefficient | One multiplier per non-zero coefficient, no sharing. |
| after symmetry sharing | FIR: symmetric pairs share one multiplier through a pre-adder. IIR: the unit `b0` is free and `b0 = b2` is shared. |
| **non-trivial** | The previous column minus coefficients that are plus or minus a power of two, which are shifts. This is the number of real multipliers. |
| distinct constants | Distinct values up to sign and power-of-two scaling. A lower bound if identical constants are shared (transposed form). |
| CSD adders | Adders needed to build the non-trivial set from shifts and adds, with no multipliers at all. Quantised only. |

In [ ]:
fd.designer()

## 2. The same thing scripted

In [ ]:
spec = fd.Spec(fs=100, fpass=10, fstop=15, ripple_db=0.1, atten_db=60)
filters = fd.design_all(spec, ["equiripple", "ls", "kaiser", "elliptic"])
for row in fd.summary(filters, spec, bits=16):
    print(f"{row['type']:<30} {row['size']:<10} non-trivial multipliers: {row['nontrivial']:<3} "
          f"ripple {row['ripple_db']:.4f} dB, stopband {row['atten_db']:.1f} dB, min bits {row['min_bits']}")

In [ ]:
fd.plot_responses(filters, spec, bits=12);

## 3. An input through the filters

RRC with roll-off 0.35 and a +10 dB adjacent channel. EVM is the ISI the filter adds to the wanted signal after a matched RRC receiver.

In [ ]:
fd.plot_signal(filters, spec, "rrc_adj", bw=14, alpha=0.35, adj_db=10);
for row in fd.summary(filters, spec, input_kind="rrc_adj", bw=14, alpha=0.35, adj_db=10, with_min_bits=False):
    print(f"{row['type']:<30} loss {row['loss_db']:.3f} dB, adjacent rejection {row['acr_db']:.1f} dB, EVM {row['evm_pct']:.3f} %")

## 4. A half-band case

Half-band needs the passband edge below fs/4 and the stopband edge above it. Every other tap is zero and the centre tap is exactly 0.5, so it needs far fewer multipliers than its tap count suggests.

In [ ]:
hb_spec = fd.Spec(fs=100, fpass=20, fstop=30, ripple_db=0.01, atten_db=80)
hb = fd.design_all(hb_spec, ["equiripple", "halfband"])
for row in fd.summary(hb, hb_spec, bits=18, with_min_bits=False):
    print(f"{row['type']:<30} {row['size']:<9} non-zero {row['direct']:<3} after symmetry {row['symmetric']:<3} non-trivial {row['nontrivial']}")

## 5. Export and use the coefficients

In [ ]:
files = fd.export(filters, spec, "filter_export", bits=16)
print("\n".join(files))
h = filters[0].h                      # float taps of the equiripple design
q = filters[0].quantized(16)          # q.q are the integers, q.frac the fractional bits
print(len(h), "taps; first integers:", q.q[:5], "with", q.frac, "fractional bits")